In [1]:
import pandas as pd

df = pd.read_excel('Obesity_Dataset/Obesity_Dataset.xlsx')

df.head()

X = df.drop(columns="Class")
y = df["Class"]

print(df.isnull().sum()) 
print(df.info())

# Valori unici dell'etichetta target => 4 => 1. Underweight 73, 2. Normal 658, 3. Overweight 592, 4. Obesity 287
print(df.Class.nunique())

Sex                                  0
Age                                  0
Height                               0
Overweight_Obese_Family              0
Consumption_of_Fast_Food             0
Frequency_of_Consuming_Vegetables    0
Number_of_Main_Meals_Daily           0
Food_Intake_Between_Meals            0
Smoking                              0
Liquid_Intake_Daily                  0
Calculation_of_Calorie_Intake        0
Physical_Excercise                   0
Schedule_Dedicated_to_Technology     0
Type_of_Transportation_Used          0
Class                                0
dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1610 entries, 0 to 1609
Data columns (total 15 columns):
 #   Column                             Non-Null Count  Dtype
---  ------                             --------------  -----
 0   Sex                                1610 non-null   int64
 1   Age                                1610 non-null   int64
 2   Height                             1610 no

L'etichetta target e' class 

In [2]:


from sklearn.model_selection import train_test_split
import numpy as np

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42, test_size=0.15)


# Non ci sono valori nulli da imputare ma comunque creo una pipeline di preprocessing nell'eventualita' di un esame
# In cui il dataset sia sparso e rumoroso

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

num_features = X.select_dtypes(include=['number']).columns
cat_features = X.select_dtypes(include=["object", "string", "bool"]).columns

"""-3 punti, non ricordavo di dover usare .columns per le features...mi dava errore e non compilava"""

num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='median')),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy='constant', fill_value="unknown")),
    ("encoder", OneHotEncoder(sparse_output=False, handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", num_pipeline, num_features),
    ("categoric", cat_pipeline, cat_features)
])

preprocessor.set_output(transform='pandas')

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)



# Selezione features tramite LDA

In [11]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis


lda = LinearDiscriminantAnalysis()
lda.fit(X_train_processed, y_train)
features_selezionate = lda.get_feature_names_out(X_train_processed.columns)
X_train_fin = pd.DataFrame(lda.transform(X_train_processed), index=X_train_processed.index, columns=features_selezionate)
X_test_fin = pd.DataFrame(lda.transform(X_test_processed), index=X_test_processed.index ,columns=features_selezionate )

# non sono sicuro sia corretto


# Classificazione con Random Forest

In [12]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

param_grid= {
    "n_estimators" : [50,100,200],
    "max_depth" : [None, 5, 10 ,15],
    "min_samples_split" : [2,5,10],
    "criterion" : ['gini', 'entropy']
}

cv = StratifiedKFold(n_splits=5,random_state=42, shuffle=True)
rfc = RandomForestClassifier()
grid = GridSearchCV(estimator=rfc,param_grid=param_grid, cv=cv, scoring='accuracy')

grid.fit(X_train_fin,y_train)



In [15]:

best_params = grid.best_params_
best_model = grid.best_estimator_
acc_rfc = best_model.score(X_test_fin, y_test)
print(best_model)
print(best_params)
print(acc_rfc)

RandomForestClassifier(max_depth=15, n_estimators=200)
{'criterion': 'gini', 'max_depth': 15, 'min_samples_split': 2, 'n_estimators': 200}
0.7272727272727273


# Rete neurale

In [ ]:
import torch
from torch import nn
import torchnn as tnn
from torch.utils.data import Dataset, DataLoader

X_train_nn, X_val_nn, y_train_nn, y_val_nn = train_test_split(X_train_fin, y_train, test_size=0.15, random_state=42, stratify=y_train)


class Data(Dataset):
    def __init__(self, X,y):
        self.X = torch.tensor(np.array(X), dtype=torch.float32)
        self.y = torch.tensor(np.array(y), dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, index):
        return self.X[index], self.y[index]

train_data = Data(X_train_nn, y_train_nn)
val_data = Data(X_val_nn, y_val_nn)
test_data = Data(X_test_fin, y_test)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Leverei almeno 3 punti, non ricordavo come impostare la rete neurale e come strutturare i layer convoluzionali
import torch.nn.functional as F
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        # Layer convoluzionale
        self.conv1 = nn.Conv1d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(32,64, kernel_size=3, padding=1)

        self.pool = nn.MaxPool1d(kernel_size=3, stride=2)
        self.dropout = nn.Dropout(0.3)
        self.fc1 = nn.Linear(64,128)
        self.fc2 = nn.Linear(128, 4)

    def forward(self, x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.con2(x)))

        # Flatten prima di passare ai fully connected -1 punto non me lo ricordavo
        x=x.view(x.size(0), - 1)
        x=F.relu(self.fc1(x))
        x=self.dropout(x)
        x=F.relu(self.fc2(x))
        return x
    print("modello caricato!")


modello caricato!


In [23]:
import torch.optim as optim


device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Net().to(device)


optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = optim.lr_scheduler.LinearLR(optimizer=optimizer )
loss_fn = nn.CrossEntropyLoss()
early_stopping = tnn.EarlyStopping(patience=5, min_delta=0.001)

train_loss, validation_loss, test_loss, accuracy, test_metrics = tnn.train_test(
    model=model,
    optimizer=optimizer,
    train_dataloader=train_loader,
    test_dataloader=test_loader,
    val_dataloader=val_loader,
    early_stopping=early_stopping,
    scheduler=scheduler,
    device=device,
    metrics=[],
    test_loss_fn=loss_fn,
    train_loss_fn=loss_fn,
    epochs=30,
    average='macro'

)

tnn.displayLosses(train_loss, test_loss, validation_loss)
tnn.displayMetrics(accuracy, test_metrics)

Epoch    1:   0%|          | 0/19 [00:00<?, ?it/s]


RuntimeError: Given groups=1, weight of size [32, 3, 3], expected input[1, 64, 3] to have 3 channels, but got 64 channels instead